# Case study: date, interaction and target-encoded features

*Session 9, block 1 practice. Author: course team, licence CC-BY-4.0.*

We add groups of features to the decision sample and measure, with a **time-based validation** (fit on 2017–2021, validate on 2022–2023, as on the leaderboard), whether each group helps. To keep the models fast and the encodings small, the target is the binary question **"is this decision in chapter 85 (electrical machinery and equipment)?"** (14.7 % of the decisions). `chapter` is used only as the target, never as an input.

Feature groups:
1. **baseline**: text statistics, language and issuing country (the inputs of leaderboard round L1),
2. **dates**: year and cyclical month of `start_date`,
3. **an interaction**: language × description length,
4. **first word**: the first word of the description, rare words grouped, then target-encoded with cross-fitting.

Theory: [page 1](../theory/01-dates-interactions-and-categories.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler, TargetEncoder

In [ ]:
def text_stats(description):
    """Simple statistics of the description of goods (Session 9, page 2), one row per decision."""
    d = description.fillna("")
    n_chars = d.str.len()
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_digits": d.str.count(r"\d"),
        "n_lines": d.str.count("\n") + 1,
        "upper_share": d.str.count(r"[A-ZÄÖÜ]") / n_chars.clip(lower=1),
        "mean_word_len": n_chars / d.str.split().str.len().clip(lower=1),
        "has_code": d.str.contains("<CODE>", regex=False).astype(int),
    }, index=description.index)

In [ ]:
df = pd.read_parquet(DATA / "train_sample.parquet")
df["first_word"] = df["description"].str.lower().str.extract(r"([^\W\d_]+)", expand=False).fillna("")
df = df.join(text_stats(df["description"]))
y = df["chapter"].eq("85").astype(int)
fit = df["start_date"] < "2022-01-01"
val = ~fit
print(fit.sum(), val.sum(), round(y[fit].mean(), 3), round(y[val].mean(), 3))

## 1. Look before you build

Group means are a cheap first check. Which of these variables relate to the target?

In [ ]:
print(y.groupby(df["start_date"].dt.year).mean().round(3).to_dict())          # by year
print(y.groupby(df["start_date"].dt.month).mean().round(3).to_dict())         # by month
print(y.groupby(df["language"]).agg(["mean", "size"]).sort_values("size", ascending=False).head(6).round(3))

## 2. The feature groups as transformers

In [ ]:
STATS = ["log_chars", "n_digits", "n_lines", "upper_share", "mean_word_len", "has_code"]


def date_features(X):
    month = X["start_date"].dt.month
    return pd.DataFrame({"year": X["start_date"].dt.year,
                         "month_sin": np.sin(2 * np.pi * month / 12),
                         "month_cos": np.cos(2 * np.pi * month / 12)}, index=X.index)


def german_x_length(X):
    """Interaction: description length for German decisions, 0 otherwise."""
    return pd.DataFrame({"de_x_log_chars": (X["language"] == "de") * X["log_chars"]}, index=X.index)


class RareGrouper(BaseEstimator, TransformerMixin):
    """Replace categories seen fewer than min_count times in the training data by 'other'."""

    def __init__(self, min_count=20):
        self.min_count = min_count

    def fit(self, X, y=None):
        counts = X.iloc[:, 0].value_counts()
        self.frequent_ = set(counts[counts >= self.min_count].index)
        return self

    def transform(self, X):
        col = X.iloc[:, 0]
        return col.where(col.isin(self.frequent_), "other").to_frame()


GROUPS = {
    "baseline": [("stats", "passthrough", STATS),
                 ("meta", OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist",
                                        sparse_output=False), ["language", "issuing_country"])],
    "dates": [("dates", FunctionTransformer(date_features), ["start_date"])],
    "interaction": [("inter", FunctionTransformer(german_x_length), ["language", "log_chars"])],
    "first word": [("word", make_pipeline(RareGrouper(5), TargetEncoder(target_type="binary",
                                                                         cv=KFold(5, shuffle=True, random_state=0))),
                    ["first_word"])],
}

## 3. Add the groups one at a time

Each model is fitted on 2017–2021 and scored on 2022–2023 with ROC AUC. A logistic regression needs the interaction column to use one; a gradient-boosting model can find interactions itself (Session 10).

In [ ]:
def validate(parts, model):
    pipe = make_pipeline(ColumnTransformer(parts), model)
    pipe.fit(df[fit], y[fit])
    return roc_auc_score(y[val], pipe.predict_proba(df[val])[:, 1])


rows, parts = [], []
for name, group in GROUPS.items():
    parts = parts + group
    rows.append({"features": "+ " + name if rows else name,
                 "logistic regression": validate(parts, make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
                 "gradient boosting": validate(parts, HistGradientBoostingClassifier(random_state=0))})
pd.DataFrame(rows).set_index("features").round(3)

**Questions.**

1. Which group brings the largest gain? Is it the same for both models?
2. Does the interaction help the logistic regression? Does it help the boosting model?
3. Why must the first-word encoder be fitted inside the pipeline (on 2017–2021 only), and why does `TargetEncoder` cross-fit even there?

## 4. The leak of naive target encoding, on validation data

Encode the first word naively (mean of the target per word over the *fitting* rows, own row included) and compare the training and validation AUC with the cross-fitted encoder.

In [ ]:
naive = y[fit].groupby(df.loc[fit, "first_word"]).mean()
f_fit = df.loc[fit, "first_word"].map(naive)
f_val = df.loc[val, "first_word"].map(naive).fillna(y[fit].mean())
enc = TargetEncoder(target_type="binary", cv=KFold(5, shuffle=True, random_state=0))
e_fit = enc.fit_transform(df.loc[fit, ["first_word"]], y[fit]).ravel()
e_val = enc.transform(df.loc[val, ["first_word"]]).ravel()
pd.DataFrame({"naive": [roc_auc_score(y[fit], f_fit), roc_auc_score(y[val], f_val)],
              "cross-fitted": [roc_auc_score(y[fit], e_fit), roc_auc_score(y[val], e_val)]},
             index=["AUC on fitting rows", "AUC on 2022-2023"]).round(3)

## Exercises

1. Tune the threshold `min_count` of `RareGrouper` (1, 5, 20, 100) with the time-based validation. Which value would you choose?
2. Replace the first word by the first *two* words. Does the cardinality grow? Does the AUC?
3. Add a count encoding of the first word (how often the word occurs in the fitting rows). Does it add anything beyond the target encoding?
4. Why would a target encoding of the first word against all 1,114 headings be impractical? What would you encode against instead?